# Colab — Unsloth LoRA 학습 (선택 경로)

- **데이터·채팅 형식**: `finetune/train_lora.py` 와 동일 (`instruction` / `input` / `output` → `example_to_text`).
- **스크립트**: `finetune/train_lora_unsloth.py` (기존 `requirements-train.txt` 만으로는 Unsloth 미포함).
- **기존 경로와 공존**: 표준 HF+PEFT만 쓰려면 `colab_train.ipynb` → `train_lora.py` 를 그대로 사용.
- **전체 절차(파일 목록·순서)**: `docs/finetune/colab/COLAB_TRAINING.md`.

**충돌 완화**: 재설치 전 `unsloth` / `unsloth_zoo` 제거 후 아래 셀 순서대로 설치. 팀에서 `--no-deps` 로 고정했다면 두 번째 설치 블록을 사용.

In [ ]:
# [A] 권장: unsloth 가 의존성까지 맞춰 설치 (Colab)
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# [B] 팀 레시피(의존성 수동) — 기존 torch/transformers 와 버전 충돌 날 때만 사용
# %pip uninstall -y unsloth unsloth-zoo
# %pip install --no-deps -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# %pip install --no-deps -q unsloth_zoo
# %pip install -q xformers trl peft accelerate bitsandbytes datasets transformers sentencepiece protobuf
pass

In [ ]:
import os

# Colab: 리포를 /content/metadata 에 두었다고 가정
PROJECT_ROOT = "/content/metadata"
FINETUNE_DIR = f"{PROJECT_ROOT}/finetune"
DATA_PATH = "/content/finetune_dataset.jsonl"
OUTPUT_DIR = "/content/lora-output-unsloth"
BASE_MODEL = "MLP-KTLim/llama-3-Korean-Bllossom-8B"

# W&B (선택): report_to wandb 시 런타임에서 로그인 또는 WANDB_API_KEY 설정
REPORT_TO = "none"  # "wandb" 로 바꾸면 train_lora_unsloth 가 run_name 전달
WANDB_RUN_NAME = "bllossom-sft-unsloth"

# Hugging Face Hub 푸시 (선택)
PUSH_TO_HUB = False
HUB_MODEL_ID = ""  # 예: "username/repo-name"

# 비밀: Colab에서는 HF_TOKEN 을 Secrets 에 넣거나 os.environ 로 설정
# os.environ["HF_TOKEN"] = "hf_..."

In [ ]:
from collections import deque
from pathlib import Path
import subprocess
import sys

script = Path(FINETUNE_DIR).resolve() / "train_lora_unsloth.py"
if not script.is_file():
    raise FileNotFoundError(f"train_lora_unsloth.py 없음: {script}")

cmd = [
    sys.executable,
    "-u",
    str(script),
    "--data", DATA_PATH,
    "--out", OUTPUT_DIR,
    "--base-model", BASE_MODEL,
    "--epochs", "3",
    "--batch-size", "2",
    "--grad-accum", "8",
    "--max-length", "2048",
    "--warmup-steps", "50",
    "--save-strategy", "steps",
    "--save-steps", "100",
    "--report-to", REPORT_TO,
    "--wandb-run-name", WANDB_RUN_NAME,
]
if PUSH_TO_HUB:
    cmd.append("--push-to-hub")
    cmd.extend(["--hub-model-id", HUB_MODEL_ID])

tail = deque(maxlen=400)
proc = subprocess.Popen(
    cmd,
    cwd=str(script.parent),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    tail.append(line)
    print(line, end="")
rc = proc.wait()
if rc != 0:
    raise RuntimeError("".join(tail))

print(f"학습 완료: {OUTPUT_DIR}")